# C4-classical-ml-practice — Practice p19 — Solution

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (60, 5))
y = rng.integers(0, 2, 60)
ks = np.arange(1, 20, 2)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=15, random_state=SEED, stratify=y
)
test_results, cv_results = [], []
for k in ks:
    candidate = KNeighborsClassifier(n_neighbors=int(k))
    candidate.fit(X_tr, y_tr)
    test_results.append(candidate.score(X_te, y_te))
    cv_results.append(cross_val_score(
        KNeighborsClassifier(n_neighbors=int(k)), X_tr, y_tr, cv=5
    ).mean())
test_accs = np.array(test_results, dtype=float)
acc_peek = float(test_accs.max())
cv_accs = np.array(cv_results, dtype=float)
k_honest = int(ks[np.argmax(cv_accs)])
honest_model = KNeighborsClassifier(n_neighbors=k_honest).fit(X_tr, y_tr)
acc_honest = float(honest_model.score(X_te, y_te))
gap = float(acc_peek - acc_honest)

test_accs, acc_peek, cv_accs, k_honest, acc_honest, gap

Each 15-row test accuracy is a noisy draw, and maximizing ten such draws preferentially selects a positive fluctuation, manufacturing the peeked 0.533 accuracy despite pure noise. The winning CV mean of about 0.644 is likewise optimistic because it is the selected maximum over ten noisy CV estimates. Only the untouched test accuracy of 0.4 is an unbiased estimate for the finally selected k = 1 model, though its small test sample makes it high-variance.

### Answer check

In [ ]:
expected_test = np.array([0.4, 7/15, 0.4, 8/15, 0.4, 7/15, 4/15, 4/15, 3/15, 4/15])
expected_cv = np.array([29/45, 21/45, 20/45, 18/45, 15/45, 16/45, 18/45, 12/45, 13/45, 13/45])
assert test_accs.shape == (10,) and np.allclose(test_accs, expected_test, atol=1e-9, rtol=0)
assert np.isclose(acc_peek, 8/15, atol=1e-9, rtol=0)
assert cv_accs.shape == (10,) and np.allclose(cv_accs, expected_cv, atol=1e-9, rtol=0)
assert k_honest == 1 and isinstance(k_honest, int)
assert np.isclose(acc_honest, 0.4, atol=1e-9, rtol=0)
assert np.isclose(gap, 2/15, atol=1e-9, rtol=0)